# 🫘 Chronic Kidney Disease (CKD) Prediction using Machine Learning

This notebook builds a **Random Forest Classifier** to predict whether a patient
has **Chronic Kidney Disease (CKD)** based on blood tests, urine tests,
and clinical observations.

---
### 📌 Dataset
- **Source:** UCI Machine Learning Repository — `Chronic_Kidney_Disease` dataset
- **Download:** https://www.kaggle.com/datasets/mansoordaku/ckdisease
- **File needed:** `kidney_disease.csv`
- **Samples:** 400 patients
- **Features:** 24 clinical features (blood tests, urine tests, comorbidities)
- **Target:** `ckd` = Chronic Kidney Disease, `notckd` = No disease

### 🧠 Model
- Algorithm: **Random Forest Classifier**
- Handles mixed data types (numeric + categorical)
- Robust to missing values after imputation

### 📋 Pipeline
1. Import Libraries
2. Load & Explore Dataset
3. Data Cleaning & Preprocessing
4. Data Visualization
5. Train-Test Split
6. Model Training
7. Evaluation
8. Feature Importance
9. Save Model

# 1. Import Libraries and Tools

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries imported successfully ✅')

# 2. Load & Explore Dataset

The dataset has **24 features** covering:
- **Urine tests:** specific gravity, albumin, sugar, red blood cells, pus cell
- **Blood tests:** blood glucose, blood urea, serum creatinine, sodium, potassium, haemoglobin
- **Clinical observations:** hypertension, diabetes, coronary artery disease, appetite

> ⚠️ This dataset has significant missing values that need careful handling.

In [ ]:
# Load the dataset
# Download from: https://www.kaggle.com/datasets/mansoordaku/ckdisease
df = pd.read_csv('kidney_disease.csv')

print('Shape:', df.shape)
print('\nColumn Names:')
print(list(df.columns))
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Check data types and non-null counts
print('Dataset Info:')
df.info()

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Columns with missing values:')
print(missing_df)

# 3. Data Cleaning & Preprocessing

The kidney disease dataset has several challenges:
- **Mixed symbols:** `?` and `\t?` represent missing values
- **Trailing tabs/spaces** in string values
- **Inconsistent class labels** (`ckd\t` vs `ckd`)
- **Mixed data types** — numerical and categorical features

Steps:
1. Clean symbols and whitespace
2. Encode categorical variables with `LabelEncoder`
3. Impute missing values with median (numerical) and mode (categorical)
4. Encode target variable

In [ ]:
# Step 1: Clean the dataframe
df.columns = df.columns.str.strip()

# Replace all forms of missing value markers
df.replace({'\t?': np.nan, '?': np.nan, '\t': ''}, regex=True, inplace=True)

# Strip whitespace from all string columns
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace('nan', np.nan)

# Fix target column inconsistencies
df['classification'] = df['classification'].replace({'ckd\t': 'ckd'})
print('Unique target values:', df['classification'].unique())

In [ ]:
# Step 2: Encode categorical columns
categorical_cols = df.select_dtypes('object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'classification']
print('Categorical columns:', categorical_cols)

le = LabelEncoder()
for col in categorical_cols:
    # Fill NaN temporarily for encoding
    df[col] = df[col].fillna('unknown')
    df[col] = le.fit_transform(df[col].astype(str))

# Encode target
df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
print('\nTarget distribution after encoding:')
print(df['classification'].value_counts())

In [ ]:
# Step 3: Convert all columns to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Step 4: Impute missing values with median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print('Missing values after imputation:', df_imputed.isnull().sum().sum())
print('Dataset ready for modelling ✅')

# 4. Data Visualization

In [ ]:
# 4.1 Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_imputed['classification'].map({1:'CKD', 0:'No CKD'}).value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#e74c3c','#3498db'], startangle=90,
            textprops={'fontsize':13})
axes[0].set_title('CKD vs No CKD Distribution', fontsize=13, fontweight='bold')

label_series = df_imputed['classification'].map({1:'CKD', 0:'No CKD'})
sns.countplot(x=label_series, palette={'CKD':'#e74c3c','No CKD':'#3498db'}, ax=axes[1])
axes[1].set_title('Patient Count by Class', fontsize=13, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x()+p.get_width()/2, p.get_height()+1), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Key clinical feature distributions by class
key_feats = ['age', 'bp', 'bgr', 'bu', 'sc', 'hemo']
key_labels = ['Age', 'Blood Pressure', 'Blood Glucose', 'Blood Urea',
              'Serum Creatinine', 'Haemoglobin']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, (feat, label) in enumerate(zip(key_feats, key_labels)):
    if feat in df_imputed.columns:
        for cls, color, name in [(1,'#e74c3c','CKD'),(0,'#3498db','No CKD')]:
            subset = df_imputed[df_imputed['classification']==cls][feat]
            axes[i].hist(subset, bins=20, alpha=0.6, color=color, label=name)
        axes[i].set_title(label, fontsize=11, fontweight='bold')
        axes[i].legend(fontsize=9)
        axes[i].set_xlabel(feat)

plt.suptitle('Clinical Feature Distributions: CKD vs No CKD',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df_imputed.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, annot_kws={'size':7})
plt.title('Feature Correlation Heatmap — Kidney Disease Dataset',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 5. Train-Test Split

In [ ]:
X = df_imputed.drop('classification', axis=1).values
y = df_imputed['classification'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')
print(f'\nClass balance (train): CKD={y_train.sum()}, No CKD={(y_train==0).sum()}')

# 6. Model Training

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Model trained ✅')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

# 7. Model Evaluation

In [ ]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print(f'Test Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC Score : {roc_auc_score(y_test, y_pred_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No CKD','CKD']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No CKD','CKD'],
            yticklabels=['No CKD','CKD'],
            ax=axes[0], cbar=False, linewidths=1)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc     = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2.5,
             label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1].set_title('ROC Curve — Kidney Disease', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Feature Importance

In [ ]:
feature_names = df_imputed.drop('classification', axis=1).columns.tolist()
importances   = model.feature_importances_
sorted_idx    = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
colors = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(feature_names))]
plt.bar(range(len(feature_names)),
        importances[sorted_idx], color=colors, edgecolor='white')
plt.xticks(range(len(feature_names)),
           [feature_names[i] for i in sorted_idx],
           rotation=45, ha='right', fontsize=9)
plt.title('Feature Importances — Kidney Disease Model', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 5 predictors:')
for i in range(5):
    print(f'  {i+1}. {feature_names[sorted_idx[i]]:<15} {importances[sorted_idx[i]]:.4f}')

# 9. Save Model

> Save to `saved_models/kidney_disease_model.sav` — expected path in `main.py`.

In [ ]:
import os
os.makedirs('saved_models', exist_ok=True)

with open('saved_models/kidney_disease_model.sav', 'wb') as f:
    pickle.dump(model, f)

print('✅ Saved: saved_models/kidney_disease_model.sav')

In [ ]:
# Verify
loaded = pickle.load(open('saved_models/kidney_disease_model.sav', 'rb'))
sample = X_test[0].reshape(1, -1)
pred   = loaded.predict(sample)
print(f'Sample prediction : {"CKD" if pred[0]==1 else "No CKD"}')
print(f'Actual label      : {"CKD" if y_test[0]==1 else "No CKD"}')
print('🎉 Model reloaded and working correctly!')